In [ ]:
%pip install -q ultralytics==8.4.1

In [ ]:
import base64
import hashlib
import json
import os
import platform
import subprocess
import time
from datetime import datetime, timezone
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile

ROOT = Path("/tmp/wastescope")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)


import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

RUN_NAME = "S0_yolo26n_seg_640"
CHECKPOINT = "best.pt"
SPLIT = "val"

IMGSZ = None  # Read the training image size from the checkpoint.
DEVICE = 0
HALF = True

AP_CONF = 0.001
CONF = 0.25
MATCH_IOU = 0.50
MAX_DET = 100
BOUNDARY_RATIO = 0.02
WARMUP = 10

DATA = ROOT / "segmentation_prepared"
REMOTE_DATA = "gdrive:WasteScope/prepared/segmentation"
REMOTE_RUNS = "gdrive:WasteScope/runs/segment"
REMOTE_REPORTS = "gdrive:WasteScope/reports/segmentation"

RCLONE_CONFIG = ROOT / "rclone.conf"

if not RCLONE_CONFIG.is_file():
    encoded = UserSecretsClient().get_secret("RCLONE_CONFIG_B64")
    RCLONE_CONFIG.write_bytes(
        base64.b64decode("".join(encoded.split()), validate=True)
    )
    del encoded

RCLONE_CONFIG.chmod(0o600)
os.environ["RCLONE_CONFIG"] = str(RCLONE_CONFIG)

RCLONE = ROOT / "bin" / "rclone"

if not RCLONE.is_file():
    request = Request(
        "https://github.com/rclone/rclone/releases/download/"
        "v1.75.2/rclone-v1.75.2-linux-amd64.zip",
        headers={"User-Agent": "curl/8.0"},
    )

    with urlopen(request, timeout=120) as response:
        payload = response.read()

    with ZipFile(BytesIO(payload)) as archive:
        member = next(
            name for name in archive.namelist()
            if Path(name).name == "rclone"
        )
        RCLONE.parent.mkdir(parents=True, exist_ok=True)
        RCLONE.write_bytes(archive.read(member))

RCLONE.chmod(0o755)

rclone_version = subprocess.run(
    [str(RCLONE), "version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()[0]

print(rclone_version)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Ultralytics:", ultralytics.__version__)